# hlb-CIFAR10 — Colab Notebook
High-level, hackable training script for CIFAR-10.

In [ ]:
# Clear notebook state to avoid bugs from re-running cells
try:
    _ = get_ipython().__class__.__name__
    %reset -f
except NameError:
    pass

## 1. Imports

In [ ]:
from functools import partial
import math
import os
import copy

import torch
import torch.nn.functional as F
from torch import nn

import torchvision
from torchvision import transforms

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

## 2. Hyperparameters
Modify these before running the rest of the notebook :
- `batchsize`: increase to use more VRAM (must stay a power of 2 ideally)
- `base_depth`: `64` (fast) or `128` (high accuracy ~95.79%)
- `train_epochs`: `1.1` for a smoke test, `12.1` for default, `90` for max accuracy
- `nb_runs`: number of full training repetitions for mean/variance statistics

In [ ]:
nb_runs   = 25     # number of repeated training runs (use 25 for proper mean/variance stats)
batchsize = 512
bias_scaler = 64

# To replicate the ~95.79%-accuracy-in-110-seconds runs, you can change the base_depth from 64->128,
# train_epochs from 12.1->90, ['ema'] epochs 10->80, cutmix_size 3->10, and cutmix_epochs 6->80
hyp = {
    'opt': {
        'bias_lr':           1.525 * bias_scaler/512,
        'non_bias_lr':       1.525 / 512,
        'bias_decay':        6.687e-4 * batchsize/bias_scaler,
        'non_bias_decay':    6.687e-4 * batchsize,
        'scaling_factor':    1./9,
        'percent_start':     .23,
        'loss_scale_scaler': 1./32,
    },
    'net': {
        'whitening': {
            'kernel_size': 2,
            'num_examples': 50000,
        },
        'batch_norm_momentum': .4,
        'cutmix_size':   3,
        'cutmix_epochs': 6,
        'pad_amount':    2,
        'base_depth':    64,  ## factor of 8 to stay tensor-core friendly
    },
    'misc': {
        'ema': {
            'epochs':       10,
            'decay_base':   .95,
            'decay_pow':    3.,
            'every_n_steps': 5,
        },
        'train_epochs':  12.1,   # set to 12.1 for the full default run
        'device':        'cuda',
        'data_location': 'data.pt',
    }
}

# Derived network depth table (computed once from base_depth)
scaler = 2.
depths = {
    'init':        round(scaler**-1 * hyp['net']['base_depth']),  # 64  @ base_depth=128
    'block1':      round(scaler** 0 * hyp['net']['base_depth']),  # 128
    'block2':      round(scaler** 2 * hyp['net']['base_depth']),  # 512
    'block3':      round(scaler** 3 * hyp['net']['base_depth']),  # 1024
    'num_classes': 10,
}

# Default conv kwargs applied globally
default_conv_kwargs = {'kernel_size': 3, 'padding': 'same', 'bias': False}

print("Hyperparameters set.")

## 3. Dataloader
Downloads CIFAR-10, normalizes it, converts to FP16, and saves to `data.pt`. subsequent runs load it instantly from disk. Only re-run this cell if `data.pt` is deleted.

In [ ]:
if not os.path.exists(hyp['misc']['data_location']):
    transform = transforms.Compose([transforms.ToTensor()])

    cifar10      = torchvision.datasets.CIFAR10('cifar10/', download=True,  train=True,  transform=transform)
    cifar10_eval = torchvision.datasets.CIFAR10('cifar10/', download=False, train=False, transform=transform)

    # Load the entire dataset as a single batch onto the GPU directly
    train_loader = torch.utils.data.DataLoader(cifar10,      batch_size=len(cifar10),      drop_last=True,  shuffle=True,  num_workers=2, persistent_workers=False)
    eval_loader  = torch.utils.data.DataLoader(cifar10_eval, batch_size=len(cifar10_eval), drop_last=True,  shuffle=False, num_workers=1, persistent_workers=False)

    train_dataset_gpu, eval_dataset_gpu = {}, {}
    train_dataset_gpu['images'], train_dataset_gpu['targets'] = [item.to(hyp['misc']['device'], non_blocking=True) for item in next(iter(train_loader))]
    eval_dataset_gpu['images'],  eval_dataset_gpu['targets']  = [item.to(hyp['misc']['device'], non_blocking=True) for item in next(iter(eval_loader))]

    # Dynamically compute mean and std from the training set
    cifar10_std, cifar10_mean = torch.std_mean(train_dataset_gpu['images'], dim=(0, 2, 3))

    def batch_normalize_images(input_images, mean, std):
        return (input_images - mean.view(1, -1, 1, 1)) / std.view(1, -1, 1, 1)

    batch_normalize_images = partial(batch_normalize_images, mean=cifar10_mean, std=cifar10_std)

    train_dataset_gpu['images'] = batch_normalize_images(train_dataset_gpu['images'])
    eval_dataset_gpu['images']  = batch_normalize_images(eval_dataset_gpu['images'])

    data = {'train': train_dataset_gpu, 'eval': eval_dataset_gpu}

    # Convert to FP16 to halve memory usage
    data['train']['images'] = data['train']['images'].half().requires_grad_(False)
    data['eval']['images']  = data['eval']['images'].half().requires_grad_(False)

    # One-hot encode labels (needed for CutMix label mixing)
    data['train']['targets'] = F.one_hot(data['train']['targets']).half()
    data['eval']['targets']  = F.one_hot(data['eval']['targets']).half()

    torch.save(data, hyp['misc']['data_location'])
    print("Dataset prepared and saved to", hyp['misc']['data_location'])
else:
    data = torch.load(hyp['misc']['data_location'])
    print("Dataset loaded from", hyp['misc']['data_location'])

# Pad images on all 4 sides for random crop augmentation
if hyp['net']['pad_amount'] > 0:
    data['train']['images'] = F.pad(data['train']['images'], (hyp['net']['pad_amount'],)*4, 'reflect')

print(f"Train images : {data['train']['images'].shape}  ({data['train']['images'].dtype})")
print(f"Eval  images : {data['eval']['images'].shape}  ({data['eval']['images'].dtype})")

## 4. Network Components
Core building blocks: `BatchNorm`, `Conv`, `Linear`, `ConvGroup`, and `FastGlobalMaxPooling`.

In [ ]:
class BatchNorm(nn.BatchNorm2d):
    def __init__(self, num_features, eps=1e-12, momentum=hyp['net']['batch_norm_momentum'], weight=False, bias=True):
        super().__init__(num_features, eps=eps, momentum=momentum)
        self.weight.data.fill_(1.0)
        self.bias.data.fill_(0.0)
        self.weight.requires_grad = weight
        self.bias.requires_grad = bias


class Conv(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        kwargs = {**default_conv_kwargs, **kwargs}
        super().__init__(*args, **kwargs)
        self.kwargs = kwargs


class Linear(nn.Linear):
    def __init__(self, *args, temperature=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.kwargs = kwargs
        self.temperature = temperature

    def forward(self, x):
        weight = self.weight * self.temperature if self.temperature is not None else self.weight
        return x @ weight.T


class ConvGroup(nn.Module):
    def __init__(self, channels_in, channels_out):
        super().__init__()
        self.channels_in  = channels_in
        self.channels_out = channels_out
        self.pool1 = nn.MaxPool2d(2)
        self.conv1 = Conv(channels_in,  channels_out)
        self.conv2 = Conv(channels_out, channels_out)
        self.norm1 = BatchNorm(channels_out)
        self.norm2 = BatchNorm(channels_out)
        self.activ = nn.GELU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.pool1(x)
        x = self.norm1(x)
        x = self.activ(x)
        x = self.conv2(x)
        x = self.norm2(x)
        x = self.activ(x)
        return x


class FastGlobalMaxPooling(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return torch.amax(x, dim=(2, 3))  # Global maximum pooling


print("Network components defined.")

## 5. Network Definition & Whitening Initialization
`SpeedyConvNet` assembles the full architecture. `make_net` builds and initializes it, including the fixed whitening convolution in the first layer.

In [ ]:
def get_patches(x, patch_shape=(3, 3), dtype=torch.float32):
    c, (h, w) = x.shape[1], patch_shape
    return x.unfold(2, h, 1).unfold(3, w, 1).transpose(1, 3).reshape(-1, c, h, w).to(dtype)


def get_whitening_parameters(patches):
    n, c, h, w = patches.shape
    est_covariance = torch.cov(patches.view(n, c*h*w).t())
    eigenvalues, eigenvectors = torch.linalg.eigh(est_covariance, UPLO='U')
    return eigenvalues.flip(0).view(-1, 1, 1, 1), eigenvectors.t().reshape(c*h*w, c, h, w).flip(0)


def set_whitening_conv(conv_layer, eigenvalues, eigenvectors, eps=1e-2, freeze=True):
    shape = conv_layer.weight.data.shape
    eigenvectors_sliced = (eigenvectors / torch.sqrt(eigenvalues + eps))[-shape[0]:, :, :, :]
    conv_layer.weight.data = torch.cat((eigenvectors_sliced, -eigenvectors_sliced), dim=0)
    if freeze:
        conv_layer.weight.requires_grad = False


def init_whitening_conv(layer, train_set=None, num_examples=None, previous_block_data=None,
                        pad_amount=None, freeze=True, whiten_splits=None):
    if train_set is not None and previous_block_data is None:
        if pad_amount > 0:
            previous_block_data = train_set[:num_examples, :, pad_amount:-pad_amount, pad_amount:-pad_amount]
        else:
            previous_block_data = train_set[:num_examples, :, :, :]

    if whiten_splits is None:
        previous_block_data_split = [previous_block_data]
    else:
        previous_block_data_split = previous_block_data.split(whiten_splits, dim=0)

    eigenvalue_list, eigenvector_list = [], []
    for data_split in previous_block_data_split:
        eigenvalues, eigenvectors = get_whitening_parameters(get_patches(data_split, patch_shape=layer.weight.data.shape[2:]))
        eigenvalue_list.append(eigenvalues)
        eigenvector_list.append(eigenvectors)

    eigenvalues  = torch.stack(eigenvalue_list,  dim=0).mean(0)
    eigenvectors = torch.stack(eigenvector_list, dim=0).mean(0)
    set_whitening_conv(layer, eigenvalues.to(dtype=layer.weight.dtype), eigenvectors.to(dtype=layer.weight.dtype), freeze=freeze)
    return layer(previous_block_data.to(dtype=layer.weight.dtype))


class SpeedyConvNet(nn.Module):
    def __init__(self, network_dict):
        super().__init__()
        self.net_dict = network_dict

    def forward(self, x):
        if not self.training:
            x = torch.cat((x, torch.flip(x, (-1,))))
        x = self.net_dict['initial_block']['whiten'](x)
        x = self.net_dict['initial_block']['activation'](x)
        x = self.net_dict['conv_group_1'](x)
        x = self.net_dict['conv_group_2'](x)
        x = self.net_dict['conv_group_3'](x)
        x = self.net_dict['pooling'](x)
        x = self.net_dict['linear'](x)
        if not self.training:
            orig, flipped = x.split(x.shape[0] // 2, dim=0)
            x = .5 * orig + .5 * flipped
        return x


def make_net():
    whiten_conv_depth = 3 * hyp['net']['whitening']['kernel_size']**2
    network_dict = nn.ModuleDict({
        'initial_block': nn.ModuleDict({
            'whiten':     Conv(3, whiten_conv_depth, kernel_size=hyp['net']['whitening']['kernel_size'], padding=0),
            'activation': nn.GELU(),
        }),
        'conv_group_1': ConvGroup(2*whiten_conv_depth, depths['block1']),
        'conv_group_2': ConvGroup(depths['block1'],    depths['block2']),
        'conv_group_3': ConvGroup(depths['block2'],    depths['block3']),
        'pooling':      FastGlobalMaxPooling(),
        'linear':       Linear(depths['block3'], depths['num_classes'], bias=False, temperature=hyp['opt']['scaling_factor']),
    })

    net = SpeedyConvNet(network_dict)
    net = net.to(hyp['misc']['device'])
    net = net.to(memory_format=torch.channels_last)
    net.train()
    net.half()

    with torch.no_grad():
        init_whitening_conv(
            net.net_dict['initial_block']['whiten'],
            data['train']['images'].index_select(0, torch.randperm(data['train']['images'].shape[0], device=data['train']['images'].device)),
            num_examples=hyp['net']['whitening']['num_examples'],
            pad_amount=hyp['net']['pad_amount'],
            whiten_splits=5000,
        )

        for layer_name in net.net_dict.keys():
            if 'conv_group' in layer_name:
                # Implicit residual: dirac init added to conv1, pure dirac for conv2
                dirac_weights_in = torch.nn.init.dirac_(torch.empty_like(net.net_dict[layer_name].conv1.weight))
                std_pre, mean_pre = torch.std_mean(net.net_dict[layer_name].conv1.weight.data)
                net.net_dict[layer_name].conv1.weight.data = net.net_dict[layer_name].conv1.weight.data + dirac_weights_in
                std_post, mean_post = torch.std_mean(net.net_dict[layer_name].conv1.weight.data)
                # Renormalize to preserve original init statistics
                net.net_dict[layer_name].conv1.weight.data.sub_(mean_post).div_(std_post).mul_(std_pre).add_(mean_pre)
                torch.nn.init.dirac_(net.net_dict[layer_name].conv2.weight)

    return net


print("Network definition and whitening helpers defined.")

## 6. Data Augmentation Helpers
GPU-side augmentations applied per-epoch: random crop, random horizontal flip, and CutMix.

In [ ]:
def make_random_square_masks(inputs, mask_size):
    if mask_size == 0:
        return None
    is_even  = int(mask_size % 2 == 0)
    in_shape = inputs.shape
    mask_center_y = torch.empty(in_shape[0], dtype=torch.long, device=inputs.device).random_(mask_size//2-is_even, in_shape[-2]-mask_size//2-is_even)
    mask_center_x = torch.empty(in_shape[0], dtype=torch.long, device=inputs.device).random_(mask_size//2-is_even, in_shape[-1]-mask_size//2-is_even)
    to_mask_y_dists = torch.arange(in_shape[-2], device=inputs.device).view(1, 1, in_shape[-2], 1) - mask_center_y.view(-1, 1, 1, 1)
    to_mask_x_dists = torch.arange(in_shape[-1], device=inputs.device).view(1, 1, 1, in_shape[-1]) - mask_center_x.view(-1, 1, 1, 1)
    to_mask_y = (to_mask_y_dists >= (-(mask_size // 2) + is_even)) * (to_mask_y_dists <= mask_size // 2)
    to_mask_x = (to_mask_x_dists >= (-(mask_size // 2) + is_even)) * (to_mask_x_dists <= mask_size // 2)
    return to_mask_y * to_mask_x


def batch_crop(inputs, crop_size):
    with torch.no_grad():
        crop_mask_batch = make_random_square_masks(inputs, crop_size)
        return torch.masked_select(inputs, crop_mask_batch).view(inputs.shape[0], inputs.shape[1], crop_size, crop_size)


def batch_flip_lr(batch_images, flip_chance=.5):
    with torch.no_grad():
        return torch.where(
            torch.rand_like(batch_images[:, 0, 0, 0].view(-1, 1, 1, 1)) < flip_chance,
            torch.flip(batch_images, (-1,)),
            batch_images
        )


def batch_cutmix(inputs, targets, patch_size):
    with torch.no_grad():
        batch_permuted    = torch.randperm(inputs.shape[0], device='cuda')
        cutmix_batch_mask = make_random_square_masks(inputs, patch_size)
        if cutmix_batch_mask is None:
            return inputs, targets
        cutmix_batch   = torch.where(cutmix_batch_mask, torch.index_select(inputs,  0, batch_permuted), inputs)
        cutmix_targets = torch.index_select(targets, 0, batch_permuted)
        portion_mixed  = float(patch_size**2) / (inputs.shape[-2] * inputs.shape[-1])
        cutmix_labels  = portion_mixed * cutmix_targets + (1. - portion_mixed) * targets
        return cutmix_batch, cutmix_labels


print("Data augmentation helpers defined.")

## 7. Training Helpers
`NetworkEMA`: exponential moving average of weights. `get_batches`: per-epoch batch iterator with augmentation. `init_split_parameter_dictionaries`: separates bias vs non-bias params for independent learning rates.

In [ ]:
class NetworkEMA(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net_ema = copy.deepcopy(net).eval().requires_grad_(False)

    def update(self, current_net, decay):
        with torch.no_grad():
            for ema_param, (param_name, incoming_param) in zip(self.net_ema.state_dict().values(), current_net.state_dict().items()):
                if incoming_param.dtype in (torch.half, torch.float):
                    ema_param.mul_(decay).add_(incoming_param.detach().mul(1. - decay))
                    if not ('norm' in param_name and 'weight' in param_name) and 'whiten' not in param_name:
                        incoming_param.copy_(ema_param.detach())

    def forward(self, inputs):
        with torch.no_grad():
            return self.net_ema(inputs)


@torch.no_grad()
def get_batches(data_dict, key, batchsize, epoch_fraction=1., cutmix_size=None):
    num_epoch_examples = len(data_dict[key]['images'])
    shuffled = torch.randperm(num_epoch_examples, device='cuda')
    if epoch_fraction < 1:
        shuffled = shuffled[:batchsize * round(epoch_fraction * shuffled.shape[0] / batchsize)]
        num_epoch_examples = shuffled.shape[0]
    crop_size = 32
    if key == 'train':
        images = batch_crop(data_dict[key]['images'], crop_size)
        images = batch_flip_lr(images)
        images, targets = batch_cutmix(images, data_dict[key]['targets'], patch_size=cutmix_size)
    else:
        images  = data_dict[key]['images']
        targets = data_dict[key]['targets']
    images = images.to(memory_format=torch.channels_last)
    for idx in range(num_epoch_examples // batchsize):
        if not (idx + 1) * batchsize > num_epoch_examples:
            yield (images.index_select(0,  shuffled[idx*batchsize:(idx+1)*batchsize]),
                   targets.index_select(0, shuffled[idx*batchsize:(idx+1)*batchsize]))


def init_split_parameter_dictionaries(network):
    params_non_bias = {'params': [], 'lr': hyp['opt']['non_bias_lr'], 'momentum': .85, 'nesterov': True, 'weight_decay': hyp['opt']['non_bias_decay'], 'foreach': True}
    params_bias     = {'params': [], 'lr': hyp['opt']['bias_lr'],     'momentum': .85, 'nesterov': True, 'weight_decay': hyp['opt']['bias_decay'],     'foreach': True}
    for name, p in network.named_parameters():
        if p.requires_grad:
            (params_bias if 'bias' in name else params_non_bias)['params'].append(p)
    return params_non_bias, params_bias


loss_fn = nn.CrossEntropyLoss(label_smoothing=0.2, reduction='none')

logging_columns_list = ['epoch', 'train_loss', 'val_loss', 'train_acc', 'val_acc', 'ema_val_acc', 'total_time_seconds']

def print_training_details(columns_list, separator_left='|  ', separator_right='  ', final='|',
                           column_heads_only=False, is_final_entry=False):
    print_string = ""
    if column_heads_only:
        for name in columns_list:
            print_string += separator_left + name + separator_right
        print_string += final
        print('-' * len(print_string))
        print(print_string)
        print('-' * len(print_string))
    else:
        for value in columns_list:
            print_string += separator_left + value + separator_right
        print_string += final
        print(print_string)
    if is_final_entry:
        print('-' * len(print_string))

print("Training helpers defined.")

## 8. Training & Evaluation Loop
The `main()` function runs one full training cycle. It trains the network, evaluates on the test set, and prints final accuracy. The training loop includes per-epoch timing and learning rate scheduling.

In [ ]:
def main():
    net_ema = None

    total_time_seconds = 0.
    current_steps      = 0.

    num_steps_per_epoch     = len(data['train']['images']) // batchsize
    total_train_steps       = math.ceil(num_steps_per_epoch * hyp['misc']['train_epochs'])
    ema_epoch_start         = math.floor(hyp['misc']['train_epochs']) - hyp['misc']['ema']['epochs']
    projected_ema_decay_val = hyp['misc']['ema']['decay_base'] ** hyp['misc']['ema']['every_n_steps']
    pct_start               = hyp['opt']['percent_start']

    net = make_net()
    non_bias_params, bias_params = init_split_parameter_dictionaries(net)

    opt      = torch.optim.SGD(**non_bias_params)
    opt_bias = torch.optim.SGD(**bias_params)

    initial_div_factor = 1e16
    final_lr_ratio     = .07
    lr_sched      = torch.optim.lr_scheduler.OneCycleLR(opt,      max_lr=non_bias_params['lr'], pct_start=pct_start, div_factor=initial_div_factor, final_div_factor=1./(initial_div_factor*final_lr_ratio), total_steps=total_train_steps, anneal_strategy='linear', cycle_momentum=False)
    lr_sched_bias = torch.optim.lr_scheduler.OneCycleLR(opt_bias, max_lr=bias_params['lr'],     pct_start=pct_start, div_factor=initial_div_factor, final_div_factor=1./(initial_div_factor*final_lr_ratio), total_steps=total_train_steps, anneal_strategy='linear', cycle_momentum=False)

    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize()

    for epoch in range(math.ceil(hyp['misc']['train_epochs'])):
        # ----- Training -----
        torch.cuda.synchronize()
        starter.record()
        net.train()

        loss_train, train_acc = None, None
        cutmix_size    = hyp['net']['cutmix_size'] if epoch >= hyp['misc']['train_epochs'] - hyp['net']['cutmix_epochs'] else 0
        epoch_fraction = 1 if epoch + 1 < hyp['misc']['train_epochs'] else hyp['misc']['train_epochs'] % 1

        for epoch_step, (inputs, targets) in enumerate(get_batches(data, key='train', batchsize=batchsize, epoch_fraction=epoch_fraction, cutmix_size=cutmix_size)):
            outputs = net(inputs)

            loss_batchsize_scaler = 512 / batchsize
            loss = loss_fn(outputs, targets).mul(hyp['opt']['loss_scale_scaler'] * loss_batchsize_scaler).sum().div(hyp['opt']['loss_scale_scaler'])

            if epoch_step % 50 == 0:
                train_acc  = (outputs.detach().argmax(-1) == targets.argmax(-1)).float().mean().item()
                train_loss = loss.detach().cpu().item() / (batchsize * loss_batchsize_scaler)

            loss.backward()
            opt.step();      opt_bias.step()
            lr_sched.step(); lr_sched_bias.step()
            opt.zero_grad(set_to_none=True); opt_bias.zero_grad(set_to_none=True)
            current_steps += 1

            if epoch >= ema_epoch_start and current_steps % hyp['misc']['ema']['every_n_steps'] == 0:
                if net_ema is None:
                    net_ema = NetworkEMA(net)
                    continue
                net_ema.update(net, decay=projected_ema_decay_val * (current_steps / total_train_steps) ** hyp['misc']['ema']['decay_pow'])

        ender.record()
        torch.cuda.synchronize()
        total_time_seconds += 1e-3 * starter.elapsed_time(ender)

        # ----- Evaluation -----
        net.eval()
        eval_batchsize = 2500
        assert data['eval']['images'].shape[0] % eval_batchsize == 0, \
            "eval_batchsize must evenly divide the eval dataset size."
        loss_list_val, acc_list, acc_list_ema = [], [], []

        with torch.no_grad():
            for inputs, targets in get_batches(data, key='eval', batchsize=eval_batchsize):
                if epoch >= ema_epoch_start:
                    ema_out = net_ema(inputs)
                    acc_list_ema.append((ema_out.argmax(-1) == targets.argmax(-1)).float().mean())
                outputs = net(inputs)
                loss_list_val.append(loss_fn(outputs, targets).float().mean())
                acc_list.append((outputs.argmax(-1) == targets.argmax(-1)).float().mean())

            val_acc     = torch.stack(acc_list).mean().item()
            ema_val_acc = torch.stack(acc_list_ema).mean().item() if epoch >= ema_epoch_start else None
            val_loss    = torch.stack(loss_list_val).mean().item()

        format_for_table = lambda x, locals: (
            f"{locals[x]}".rjust(len(x)) if type(locals[x]) == int
            else "{:0.4f}".format(locals[x]).rjust(len(x)) if locals[x] is not None
            else " " * len(x)
        )
        print_training_details(
            list(map(partial(format_for_table, locals=locals()), logging_columns_list)),
            is_final_entry=(epoch >= math.ceil(hyp['misc']['train_epochs'] - 1))
        )

    return ema_val_acc


print("main() defined — ready to train.")

## 9. Run Training
Executes `nb_runs` independent training runs and reports mean accuracy and variance. With `nb_runs=1` variance is 0 by definition (correction=0). Set `nb_runs=25` to reproduce the official benchmark statistics.

In [ ]:
print_training_details(logging_columns_list, column_heads_only=True)

acc_list = []
for run_num in range(nb_runs):
    print(f"\n--- Run {run_num + 1}/{nb_runs} ---")
    acc_list.append(torch.tensor(main()))

acc_tensor = torch.stack(acc_list)
mean_acc   = torch.mean(acc_tensor).item()
var_acc    = torch.var(acc_tensor, correction=(0 if nb_runs == 1 else 1)).item()
print(f"\nFinal results over {nb_runs} run(s):")
print(f"  Mean ema_val_acc : {mean_acc:.4f}")
print(f"  Variance         : {var_acc:.6f}")